#Transformer-Based Classification with Rules

## Query Analysis
The script first processes the search query using rule-based analysis.
It leverages a set of pre-compiled regular expressions to scan the query text for keywords and patterns associated with different intent types.
This initial analysis provides a quick, heuristic categorization of the query, which serves as one of the inputs for further classification.

## SERP Features Analysis
The script also uses data extracted from the Search Engine Results Page (SERP).
It processes a string of SERP features and maps each feature to a corresponding intent category.
By counting the occurrences of each SERP feature and computing percentages, the script generates additional signals that help determine the likely intent behind the search query.

## Transformer Model
A transformer-based zero-shot classification model from Hugging Face is employed to produce the final intent classification.
This model takes the combined input—which includes the original query, the rule-based label, and the SERP data analysis—and outputs the final intent label.
The transformer provides a more nuanced decision compared to rule-based methods alone, especially when the intent is not immediately clear.

## Input Requirements
The script requires a CSV file that contains at least two columns: one for search queries and one for SERP features.
The expected column names are "Keyword" for the query text and "SERP Features" for the associated SERP data.
Any missing or NaN values in these columns are replaced with empty strings to ensure smooth processing.

## Script Processing Enhancements Applied
The script is optimized for efficient processing in several ways.
Regular expressions are pre-compiled to speed up the rule-based analysis.
Batch processing is implemented so that multiple queries are classified in one transformer inference call, reducing overhead.
The transformer model is loaded lazily and configured to use a GPU if available, which further accelerates processing when handling large datasets.

In [ ]:
import re
import pandas as pd
import time
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from google.colab import files
from transformers import pipeline

class SearchIntentClassifier:
    """A classifier that determines search intent based on query text and SERP features."""

    # All intent categories (used for rule-based and reporting)
    INTENT_CATEGORIES = ["Informational", "Transactional", "Commercial", "Navigational", "Unclassified"]

    def __init__(self, model_name: str = "facebook/bart-large-mnli", batch_size: int = 16):
        """
        Initialize the classifier with pattern dictionaries and transformer model.

        Args:
            model_name: The transformer model to use for zero-shot classification
            batch_size: Batch size for processing multiple queries at once
        """
        self.batch_size = batch_size

        # Initialize pattern dictionaries for rule-based classification
        self._init_patterns()

        # Initialize SERP feature mapping
        self._init_serp_mapping()

        # Define transformer candidates (exclude "Unclassified")
        self.transformer_candidates = ["Informational", "Transactional", "Commercial", "Navigational"]

        # Initialize transformer model (lazy loading)
        self.classifier = None
        self.model_name = model_name

    def _init_patterns(self):
        """Initialize regex patterns for each intent category."""
        self.patterns = {
            "Informational": r"\b(?:how|why|does|which|when|who|guide|tutorial|learn|examples|resource|ideas|tips|difference between|strategies|ways|example|directory|news|basics|concepts|troubleshoot|resolve|risks|beginners|upgrade|improve|optimize|case study|best practices|templates|pros and cons|analysis|articles|audit|benefits|blogs|build|calculate|campaign|choose|common questions|consider|cost|create|definition|definitive guide|earn|easy steps|essential|events|explainer video|factors|find|fix|grow|hacks|how to|ideal|importance of|increase|infographic|information|meaning|measure|metrics|mistakes|myths|opportunities|popular|practices|questions|remove|set up|shortcuts|stats|strategy|study|successful|tactics|techniques|terms|things to know|time|trends|tricks|types|updates|well explained|tutorials|framework|checklist|process|plan|test|glossary|sample|survey|exercises|report|certification|webinar|calendar|quiz|directories|to do list|design basics)\b",

            "Transactional": r"\b(?:buy|purchase|order|price|quote|for sale|sale|deal|offers|discount|coupon|promo code|voucher|bargain|cost|pricing|cheap|affordable|expensive|premium|near me|online|e-commerce|shopping cart|checkout|shop|store|retailer|dealer|supplier|marketplace|auction|bid|buy now|pay|payment|finance|leasing|rent|rental|subscribe|subscription|free trial|book|reserve|enquire|inquiry|quote|estimate|cost calculator|shipping|delivery|available|in stock|out of stock|pre-order|backorder|special offer|limited time offer|flash sale|clearance|wholesale|bulk buy|group buy|membership|reward|loyalty|gift card|cashback|trade-in|upgrade|renew|secure checkout)\b",

            "Commercial": r"\b(?:best|top|vs|review|compare|comparison|ratings|recommendation|buyer's guide|alternative|feature|benefit|cost-effective|affordable|premium|quality|deal|offers|savings|discount|value|choice|selection|rankings|user ratings|testimonials|feedback|pros|cons|evaluation|assessment|analysis|benchmark|brand|models|versions|upgrade|update|launch|new|latest|innovative|technology|solutions|providers|vendors|suppliers|applications|software|platform|systems|tools|equipment|machinery|accessories|devices|gadgets|electronics|appliances|products|materials|suppli|kit|package|bundle)\b",

            "Navigational": r"\b(?:login|register|homepage|website|account|portal|open|access|find|locate|contact us|help desk|FAQ|support|customer service|map|location|directions|hours|site|navigate|URL|service center|branch|office|near me|address)\b"
        }

        # Precompile regex patterns for better performance
        self.compiled_patterns = {
            category: re.compile(pattern, re.IGNORECASE)
            for category, pattern in self.patterns.items()
        }

    def _init_serp_mapping(self):
        """Initialize the mapping of SERP features to intent categories."""
        self.serp_mapping = {
            "Adwords bottom": "Transactional",
            "AdWords top": "Transactional",
            "Bottom ads": "Transactional",
            "Carousel": "Commercial",
            "FAQ": "Informational",
            "Featured snippet": "Informational",
            "Image": "Informational",
            "Image pack": "Informational",
            "Instant answer": "Informational",
            "Jobs search": "Transactional",
            "Knowledge panel": "Informational",
            "Local pack": "Navigational",
            "Local teaser pack": "Navigational",
            "People also ask": "Informational",
            "Reviews": "Commercial",
            "Shopping ads": "Transactional",
            "Shopping results": "Transactional",
            "Site links": "Navigational",
            "Sitelinks": "Navigational",
            "Thumbnail": "Informational",
            "Top ads": "Transactional",
            "Top stories": "Informational",
            "Tweet": "Informational",
            "Twitter": "Informational",
            "Video": "Informational",
            "Video Carousel": "Informational",
            "Video preview": "Informational",
            "Videos": "Informational",
            "AI Overview": "Informational",
            "Answer Box": "Informational",
            "App": "Transactional",
            "Bottom Ads": "Transactional",
            "Carousel results": "Commercial",
            "Commercial Units": "Transactional",
            "Copyright infringement": "Unclassified",
            "Currency Box": "Informational",
            "Events": "Informational",
            "Explore Brands": "Navigational",
            "Featured Snippet": "Informational",
            "Featured snippets": "Informational",
            "Find Results On": "Informational",
            "Flights": "Transactional",
            "GMB": "Navigational",
            "Google Flights": "Transactional",
            "Google Hotels": "Transactional",
            "Google Posts": "Informational",
            "Google Reviews": "Commercial",
            "Hotel pack": "Transactional",
            "Images": "Informational",
            "Jobs": "Transactional",
            "Knowledge card": "Informational",
            "Knowledge graph": "Informational",
            "Local Services": "Transactional",
            "Map": "Navigational",
            "Math Solver": "Informational",
            "Mention Carousel": "Informational",
            "Multi Carousel": "Commercial",
            "Organic Results": "Informational",
            "Paid Results": "Transactional",
            "People Also Search For": "Informational",
            "Podcasts": "Informational",
            "Popular Products": "Transactional",
            "positions.title.datasets": "Informational",
            "positions.title.discussions_and_forums": "Informational",
            "Product Consideration": "Commercial",
            "Questions & Answers": "Informational",
            "Recipes": "Informational",
            "Refine Products": "Transactional",
            "Related searches": "Informational",
            "Scholarly Articles": "Informational",
            "See results about": "Informational",
            "Shopping": "Transactional",
            "Sites": "Navigational",
            "Sports": "Informational",
            "Stocks box": "Informational",
            "Things to Do": "Informational",
            "Top Ads": "Transactional",
            "Top Sights": "Navigational",
            "Videos Block": "Informational",
            "Visual Stories": "Informational",
            "X (ex-Twitter)": "Informational"
        }

        # Precompile SERP feature patterns for faster matching
        self.compiled_serp_patterns = {
            feature: re.compile(re.escape(feature), re.IGNORECASE)
            for feature in self.serp_mapping.keys()
        }

    def _load_model(self):
        """Lazy loading of the transformer model with GPU support if available."""
        if self.classifier is None:
            print("Loading transformer model. This may take a moment...")
            start_time = time.time()
            device = 0 if torch.cuda.is_available() else -1
            self.classifier = pipeline(
                "zero-shot-classification",
                model=self.model_name,
                device=device
            )
            print(f"Model loaded in {time.time() - start_time:.2f} seconds.")

    def rule_based_intent(self, query: str) -> str:
        """
        Determine the intent categories based on rule-based patterns in the query.

        Args:
            query: The search query text

        Returns:
            A comma-separated string of matched intent categories, or "Unclassified" if none match
        """
        if not query or not isinstance(query, str):
            return "Unclassified"

        categories = []
        for category, pattern in self.compiled_patterns.items():
            if pattern.search(query):
                categories.append(category)

        return ", ".join(sorted(set(categories))) if categories else "Unclassified"

    def calculate_serp_percentages(self, serp_features: str) -> Dict[str, float]:
        """
        Calculate the percentage of SERP features that map to each intent category.

        Args:
            serp_features: String containing SERP features separated by delimiters

        Returns:
            Dictionary with percentage values for each intent category
        """
        if not isinstance(serp_features, str):
            return {category: 0 for category in self.INTENT_CATEGORIES}

        counts = {category: 0 for category in self.INTENT_CATEGORIES}
        total = 0

        for feature, pattern in self.compiled_serp_patterns.items():
            cnt = len(pattern.findall(serp_features))
            if cnt > 0:
                intent = self.serp_mapping[feature]
                counts[intent] += cnt
                total += cnt

        if total > 0:
            percentages = {cat: (count / total) * 100 for cat, count in counts.items()}
        else:
            percentages = {cat: 0 for cat in counts}

        return percentages

    def batch_classify(self, queries: List[str], serp_features_list: List[str]) -> List[Dict]:
        """
        Classify multiple queries in batches for better performance.

        Args:
            queries: List of search query texts
            serp_features_list: List of SERP features strings

        Returns:
            List of dictionaries containing classification results
        """
        # Ensure model is loaded
        self._load_model()

        results = []
        for i in range(0, len(queries), self.batch_size):
            batch_queries = queries[i:i+self.batch_size]
            batch_serp = serp_features_list[i:i+self.batch_size]

            # Precompute the rule-based labels, SERP percentages, and combined input texts for the batch
            input_texts = []
            precomputed = []
            for query, serp in zip(batch_queries, batch_serp):
                rule_label = self.rule_based_intent(query)
                serp_percents = self.calculate_serp_percentages(serp)
                percentages_str = ", ".join([f"{k}: {v:.1f}%" for k, v in serp_percents.items()])
                combined_input = (f"Query: {query} | Rule-based Label: {rule_label} | "
                                  f"SERP Features: {serp} | SERP Percentages: {percentages_str}")
                input_texts.append(combined_input)
                precomputed.append((query, serp, rule_label, serp_percents, combined_input))

            # Run transformer inference on the entire batch using only the desired candidates
            zero_shot_results = self.classifier(input_texts, self.transformer_candidates, batch_size=self.batch_size)

            for result, (query, serp, rule_label, serp_percents, combined_input) in zip(zero_shot_results, precomputed):
                final_label = result["labels"][0]
                results.append({
                    "Keyword": query,
                    "SERP Features": serp,
                    "Rule-based Label": rule_label,
                    "SERP Percentages": serp_percents,
                    "Final Intent": final_label,
                    "Combined Input": combined_input
                })

            print(f"Processed {len(results)}/{len(queries)} queries...")

        return results


def process_csv_file(classifier: SearchIntentClassifier, file_path: str = None) -> pd.DataFrame:
    """
    Process a CSV file containing search queries and their SERP features.

    Args:
        classifier: An instance of SearchIntentClassifier
        file_path: Path to the CSV file (if None, will prompt for upload)

    Returns:
        DataFrame containing the classification results
    """
    if file_path is None:
        print("Please upload your CSV file (with columns 'Keyword' and 'SERP Features'):")
        uploaded = files.upload()
        file_path = list(uploaded.keys())[0]

    # Load and validate data
    try:
        data = pd.read_csv(file_path)
        required_columns = ["Keyword", "SERP Features"]
        missing_columns = [col for col in required_columns if col not in data.columns]

        if missing_columns:
            raise ValueError(f"Missing required columns: {', '.join(missing_columns)}")

        # Handle NaN values
        data["Keyword"] = data["Keyword"].fillna("")
        data["SERP Features"] = data["SERP Features"].fillna("")

    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return pd.DataFrame()

    print(f"Processing {len(data)} queries...")

    # Batch classify all queries
    results = classifier.batch_classify(
        data["Keyword"].tolist(),
        data["SERP Features"].tolist()
    )

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Expand SERP percentages into separate columns for easier analysis
    serp_percent_columns = pd.json_normalize(results_df["SERP Percentages"])
    for col in serp_percent_columns.columns:
        results_df[f"SERP %_{col}"] = serp_percent_columns[col]

    # Drop the dictionary column
    results_df = results_df.drop("SERP Percentages", axis=1)

    return results_df


#########################################
# 5. Visualization Functions
#########################################
def visualize_intent_distribution(df):
    """
    Create visualizations for intent distribution.

    Args:
        df (pd.DataFrame): DataFrame with classification results
    """
    intent_col = 'Final Intent'

    # Count the occurrences of each intent
    intent_counts = df[intent_col].value_counts()

    # Create a figure with a pie chart and a bar chart
    plt.figure(figsize=(12, 6))

    # Pie chart
    plt.subplot(1, 2, 1)
    intent_counts.plot.pie(autopct='%1.1f%%', startangle=90,
                             colors=sns.color_palette("viridis", len(intent_counts)))
    plt.ylabel('')
    plt.title('Intent Distribution')

    # Bar chart
    plt.subplot(1, 2, 2)
    ax = intent_counts.plot.bar(color=sns.color_palette("viridis", len(intent_counts)))
    plt.title('Intent Counts')
    plt.ylabel('Count')
    plt.xlabel('Intent')

    # Add count labels to the bars
    for i, v in enumerate(intent_counts):
        ax.text(i, v + 0.1, str(v), ha='center')

    plt.tight_layout()
    plt.show()


def main():
    """Main function to run the classification process."""
    # Create classifier
    classifier = SearchIntentClassifier()

    # Process CSV file
    results_df = process_csv_file(classifier)

    if not results_df.empty:
        # Save results
        output_filename = "search_intent_classifications.csv"
        results_df.to_csv(output_filename, index=False)

        # Download results
        try:
            files.download(output_filename)
            print(f"Results saved to {output_filename} and downloaded.")
        except Exception as e:
            print(f"Error downloading file: {e}")
            print(f"Results saved to {output_filename}.")

        # Print summary
        print("\nIntent Classification Summary:")
        print(results_df["Final Intent"].value_counts())

        # Visualize intent distribution
        visualize_intent_distribution(results_df)


if __name__ == "__main__":
    main()
